In [ ]:
# Análise de Lucratividade de Rotas Aéreas

## Objetivo

Este projeto tem como objetivo analisar a rentabilidade operacional de rotas aéreas utilizando PySpark no Databricks.

A partir de um conjunto de dados público contendo informações sobre receitas, custos operacionais, demanda e características dos voos, busca-se identificar os principais fatores que influenciam a lucratividade das operações aéreas.

As principais questões de negócio investigadas são:

- Quais fatores estão mais associados à lucratividade das rotas?
- Como a estrutura de custos impacta o resultado financeiro?
- Qual a relação entre taxa de ocupação (Load Factor) e margem de lucro?
- Como o tipo de aeronave influencia a eficiência operacional?

Além da análise exploratória, o projeto demonstra um fluxo de preparação e enriquecimento de dados utilizando PySpark em ambiente Databricks, simulando um processo típico de análise de dados em ambiente corporativo.

In [0]:
import pandas as pd
from pyspark.sql.functions import col, count, when, avg
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
# =============================================================================
# LEITURA DE DADOS 
# =============================================================================

# FONTE: https://www.kaggle.com/datasets/waleedfaheem/airline-route-profitability-and-cost-analysis

TABLE_NAME = "default.airline_route_profitability"

try:

    # -------------------------------------------------------------------------
    # Leitura da tabela no catálogo do Databricks
    # -------------------------------------------------------------------------

    df = spark.table(TABLE_NAME)

    print("✅ Tabela '{TABLE_NAME}' carregada com sucesso!")

    # -------------------------------------------------------------------------
    # Informações iniciais
    # -------------------------------------------------------------------------
    
    total_rows = df.count()
    print(f"Total de registros: {total_rows}")

    display(df.limit(5))

except Exception as e:
    print(f"❌ Erro ao carregar a tabela '{TABLE_NAME}'")
    print(f"Detalhes do erro: {e}")

In [0]:
df.printSchema() # analisando os tipos de variáveis



In [0]:
display(df.describe()) # estatísticas descritivas

In [0]:
print(df.columns) # lista com todas as colunas

In [0]:
print(f"Linhas totais: {total_rows}") 
print(f"Linhas distintas: {df.distinct().count()}") # verificando se há linhas duplicadas

In [0]:
from pyspark.sql.functions import col, count, when  # verificando observações vazias

display(
    df.select([
        count(
            when(col(c).isNull(), c)
        ).alias(c)
        for c in df.columns
    ])
)

In [0]:
from pyspark.sql.functions import when  # classificando lucratividade das rotas

df = df.withColumn(
    "Profitability",
    when(df.Profit_Margin >= 20, "High")
    .when(df.Profit_Margin >= 10, "Medium")
    .when(df.Profit_Margin >= 0, "Low")
    .otherwise("Loss")
)

In [0]:
df = df.withColumn(  # coluna de lucro por hora
    "Profit_Per_Hour",
    when(df.Flight_Hours > 0,
    df.Profit / df.Flight_Hours)
)

In [0]:
df = df.withColumn(  # coluna de receita por passageiro
    "Revenue_Per_Passenger",
    when(df.Passengers > 0,
    df.Total_Revenue / df.Passengers)
)

In [0]:
df = df.withColumn(  # coluna de custo por passageiro
    "Cost_Per_Passenger",
    when(df.Passengers > 0,
    df.Total_Cost / df.Passengers)
)

In [0]:
df = df.withColumn(  # coluna de lucro por passageiro
    "Profit_Per_Passenger",
    when(df.Passengers > 0,
    df.Profit / df.Passengers)
)

In [0]:
df.write.mode("overwrite").saveAsTable(  # salvando o dataframe com as colunas criadas
    "default.airline_route_profitability_enriched"
)

In [0]:
display(df.groupBy("Route").avg("Profit").orderBy(col("avg(Profit)").desc()))  # lucro médio por rota em ordem decrescente

In [0]:
display(df.groupBy("Route").avg("Profit_Margin").orderBy(col("avg(Profit_Margin)").desc()).limit(10))  # lucro médio percentual por rota em ordem decrescente

In [0]:
display(df.groupBy("Aircraft_Type").avg("Profit").orderBy(col("avg(Profit)").desc()))  # lucro médio por modelo de aeronave


In [0]:
display(df.groupBy("Demand_Level").avg("Profit_Margin"))  # lucro médio por nível de demanda

In [0]:
display(df.groupBy("Season").agg({"Profit":"avg",         # lucro e quantidade de passageiros médios por temporada
                                  "Passengers":"avg"}))

In [0]:
display(df.groupBy("Route_Category").agg(                 # percentual do custo do combustível por categoria de rota
avg((col("Fuel_Cost") / col("Total_Cost")) * 100)
)
)


In [0]:
display(df.groupBy("Profitability").agg(                  # lucratividade e custo médio proporcional de combustível
    avg((col("Fuel_Cost") / col("Total_Cost")) * 100)
)
)

In [0]:
display(df.groupBy("Profitability").avg("Load_Factor").orderBy(col("avg(Load_Factor)").desc())) # lucratividade e load factor médio

In [0]:
display(df.groupBy("Profitability").avg("Revenue_Per_Passenger").orderBy(col("avg(Revenue_Per_Passenger)").desc()))  # lucratividade e receita média por passageiro

In [0]:
display(                                               # lucratividade com lucro médio, load factor médio, receita e custo por passageiro médios e lucro médio por hora
    df.groupBy("Profitability")
      .avg(
          "Profit",
          "Load_Factor",
          "Revenue_Per_Passenger",
          "Cost_Per_Passenger",
          "Profit_Per_Hour"
      ).orderBy(col("avg(Profit)").desc()
)
)

In [0]:
display(df.groupBy("Aircraft_Type").avg("Profit_Per_Hour", "Profit_Margin", "Load_Factor").orderBy(col("avg(Profit_Per_Hour)").desc()))  # média de lucro por hora, lucro percentual e load factor por modelo de aeronave

In [0]:
display(df.groupBy("Aircraft_Type").avg("Revenue_Per_Passenger", "Cost_Per_Passenger", "Profit_Per_Passenger").orderBy(col("avg(Revenue_Per_Passenger)").desc())  # média de receita, custo e lucro por passageiro para cada modelo de aeronave
)

In [0]:
display(                                              # lucro percentual por modelo de aeronave e categoria de rota
    df.groupBy("Route_Category", "Aircraft_Type")
      .avg("Profit_Margin")
      .orderBy(
          col("Route_Category"),
          col("avg(Profit_Margin)").desc()
      )
)

In [0]:
display(                                               # médias de receita por passageiro, lucro por hora e lucro percentual por categoria de rota 
    df.groupBy("Route_Category")
      .avg(
          "Revenue_Per_Passenger",
          "Profit_Per_Hour",
          "Profit_Margin"
      )
)

In [0]:
# =============================================================================
# MATRIZ DE CORRELAÇÃO
# =============================================================================

# Colunas com métricas de negócio

business_cols = [
    "Load_Factor",
    "Flight_Hours",
    "Passengers",
    "Total_Revenue",
    "Total_Cost",
    "Profit",
    "Profit_Margin",
    "Profit_Per_Hour"
]

# Converter para Pandas e calcular a matriz de correlação	

business_corr = (
    df
    .select(business_cols)
    .toPandas()
    .corr()
)


# Plotar a matriz de correlação
plt.figure(figsize=(8, 6))

sns.heatmap(
    business_corr,
    annot=True,
    fmt=".2f",
    cmap="RdYlBu_r",
    center=0
)

plt.title("Business Metrics Correlation")
plt.tight_layout()
plt.show()
# =============================================================================
# 

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import avg

# Lista das colunas de custo
cost_columns = [
    "Fuel_Cost",
    "Maintenance_Cost",
    "Crew_Cost",
    "Depreciation_Cost",
    "Insurance_Cost",
    "Airport_Fees",
    "Catering_Cost",
    "Handling_Cost",
    "Navigation_Fees",
    "Sales_Distribution_Cost",
    "Passenger_Service_Cost",
    "Overhead_Cost",
    "Marketing_Cost",
    "IT_Systems_Cost"
]

# Calcular o percentual médio de cada custo em relação ao custo total
results = []

for cost_col in cost_columns:
    
    avg_pct = (
        df
        .select(((df[cost_col] / df["Total_Cost"]) * 100).alias("pct"))
        .agg(avg("pct").alias("avg_pct"))
        .collect()[0]["avg_pct"]
    )

    results.append(
        Row(
            Cost_Category=cost_col,
            Avg_Percentage=round(avg_pct, 2)
        )
    )

# Criar DataFrame Spark
cost_pct_df = spark.createDataFrame(results)

# Ordenar do maior para o menor percentual
cost_pct_df = cost_pct_df.orderBy(
    col("Avg_Percentage").desc()
)

# Salvar
cost_pct_df.write.mode("overwrite").saveAsTable(
    "default.cost_structure_analysis"
)

display(cost_pct_df)

In [ ]:
# Conclusões

A análise exploratória permitiu identificar alguns fatores relevantes relacionados à lucratividade das operações aéreas presentes no conjunto de dados analisado.

## Principais resultados

### 1. A receita por passageiro é o principal fator associado à lucratividade

As rotas classificadas como **High** apresentaram uma receita média por passageiro aproximadamente 2,5 vezes superior às rotas classificadas como **Loss**, indicando que o aumento da receita exerce maior influência sobre o lucro do que pequenas reduções de custo.

---

### 2. A taxa de ocupação contribui para melhores resultados, mas não explica sozinha a lucratividade

Foi observada uma correlação positiva entre **Load Factor** e **Profit Margin**, indicando que voos com maior ocupação tendem a apresentar margens superiores. Entretanto, essa relação é moderada, sugerindo que outros fatores, como receita por passageiro e estrutura de custos, possuem papel igualmente importante.

---

### 3. Aeronaves de grande porte apresentaram maior eficiência financeira

No conjunto de dados analisado, aeronaves como Airbus A380 e Boeing 777-300ER apresentaram maiores valores de lucro por hora de voo e margem de lucro. A análise indica que esse desempenho está relacionado principalmente à maior receita gerada por passageiro, compensando os custos operacionais mais elevados.

---

### 4. A estrutura de custos é concentrada em poucas categorias

Os maiores componentes do custo operacional foram:

- Sales Distribution Cost
- Fuel Cost
- Overhead Cost
- Depreciation Cost

Essas categorias representam a maior parcela do custo total das operações presentes no dataset.

---

## Considerações finais

Os resultados obtidos demonstram como técnicas de análise exploratória e feature engineering podem auxiliar na compreensão dos fatores que impactam a rentabilidade das operações aéreas.

Além dos insights de negócio, o projeto evidencia a utilização de ferramentas amplamente empregadas em ambientes corporativos, como Databricks e PySpark, contemplando etapas de ingestão, validação, transformação e análise de dados.

Por se tratar de um dataset público destinado a fins educacionais, os resultados devem ser interpretados como uma simulação analítica, não representando necessariamente a estrutura operacional ou financeira de uma companhia aérea específica.